# Aplicación del Modelo Final

Con nuestro modelo final ya entrenado hacemos las predicciones y las guardamos en un csv.

## Dependencias

In [1]:
import joblib
import pickle
import pandas as pd

## Cargamos Datos Predicción

In [3]:
# Abre el archivo en modo lectura binaria ('rb')
with open('../data/bank_competition.pkl', 'rb') as archivo:
    df = pickle.load(archivo)

/tmp/ipykernel_138841/2127464985.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df = pickle.load(archivo)


## Definción funciones Preprocesado

Introducimos las funciones custom del preprocesado en este entorno para que el modelo funcione.

In [11]:
# Pdays preprocessing
def pdays_transform(x):
    if x == -1:
        return 0
    elif x < 100:
        return 1
    elif x < 200:
        return 2
    elif x < 400:
        return 3
    else:
        return 4

# Función para el preprocesado de los pdays
def process_pdays(X: pd.DataFrame) -> pd.DataFrame:
    X_out = X.copy()
    X_out['wasContacted'] = X_out['pdays'].map(lambda x: 0 if x == -1 else 1)
    X_out['pdaysTransformed'] = X_out['pdays'].map(pdays_transform)
    return X_out

# Poutcome preprocessing
def calculate_propensity(row):
    if row['poutcome'] != 'success':
        return 0
    elif row['pdays'] < 0:
        return 0
    else:
        return 1/(1 + row['pdays'])

# Función para el preprocesado de pooutcome (crear interacción con pdays)
def process_poutcome(X: pd.DataFrame) -> pd.DataFrame:
    X_out = X.copy()
    # Creamos una interacción entre poutcome_success y pdays
    X_out['DepositPropensity'] = X_out[['poutcome', 'pdays']].apply(calculate_propensity, axis=1)
    return X_out

# Función para imputar los Missing Values de marital
def process_marital(X):
    X_out = X.copy()
    X_out['marital'] = X_out['marital'].fillna('unknown')
    return X_out

# Función para eliminar las columnas innecesarias
def drop_columns(X):
    return X.drop(['pdays'], axis=1, errors='ignore')

## Importamos el Modelo

In [16]:
final_model = joblib.load('../models/modelo_final.joblib')

## Predicciones

Ahora que ya tenemos los datos de competición cargados y el modelo final hacemos las predicciones y las guardamos en un csv.

In [17]:
# Hacemos las predicciones
predictions = final_model.predict(df)

# Remapeamos numeros a yes/no?

# Creamos un Dataframe
df_resultados = pd.DataFrame({
    'deposit': predictions
})

# Guardamos el csv
df_resultados.to_csv('../results/predicciones.csv', index=False)